In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")

In [3]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [4]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [5]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [6]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [7]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


{'topic': 'pizza',
 'joke': [{'type': 'text',
   'text': 'Why does a pizza never go out on a date?\n\nBecause it’s already got a **pizza** my heart, but it’s afraid of getting **cheesy**!',
   'extras': {'signature': 'EnEKbwFpFH0TxkToFjXaeu9cbVNuCGFw6IsJ9MWsIcvLRWoJU+QzQW+1kpcSqw7CLIoCBdUscLV946JgSGPTXYaPvMkjERMLH68u4FVo3zTNK5hlOMzV8XBxl26IUfs2vD6QqBcArOmTX3t92exPzNNZQg=='}}],
 'explanation': [{'type': 'text',
   'text': 'This joke relies on a play on words (puns) involving pizza-related terms. Here is the breakdown:\n\n**1. "A pizza my heart"**\nThis is a pun on the common romantic phrase, **"a piece of my heart."** By using the word "pizza," the joke implies that the speaker is already romantically attached to the pizza, or that the pizza is "taken."\n\n**2. "Afraid of getting cheesy"**\nThis works on two levels:\n*   **The Literal Meaning:** Pizza is covered in cheese, and when it gets hot, it gets stringy or "cheesy."\n*   **The Figurative Meaning:** In English, if someone is being

In [8]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'Why does a pizza never go out on a date?\n\nBecause it’s already got a **pizza** my heart, but it’s afraid of getting **cheesy**!', 'extras': {'signature': 'EnEKbwFpFH0TxkToFjXaeu9cbVNuCGFw6IsJ9MWsIcvLRWoJU+QzQW+1kpcSqw7CLIoCBdUscLV946JgSGPTXYaPvMkjERMLH68u4FVo3zTNK5hlOMzV8XBxl26IUfs2vD6QqBcArOmTX3t92exPzNNZQg=='}}], 'explanation': [{'type': 'text', 'text': 'This joke relies on a play on words (puns) involving pizza-related terms. Here is the breakdown:\n\n**1. "A pizza my heart"**\nThis is a pun on the common romantic phrase, **"a piece of my heart."** By using the word "pizza," the joke implies that the speaker is already romantically attached to the pizza, or that the pizza is "taken."\n\n**2. "Afraid of getting cheesy"**\nThis works on two levels:\n*   **The Literal Meaning:** Pizza is covered in cheese, and when it gets hot, it gets stringy or "cheesy."\n*   **The Figurative Meaning:** In English, if someon

In [9]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'Why does a pizza never go out on a date?\n\nBecause it’s already got a **pizza** my heart, but it’s afraid of getting **cheesy**!', 'extras': {'signature': 'EnEKbwFpFH0TxkToFjXaeu9cbVNuCGFw6IsJ9MWsIcvLRWoJU+QzQW+1kpcSqw7CLIoCBdUscLV946JgSGPTXYaPvMkjERMLH68u4FVo3zTNK5hlOMzV8XBxl26IUfs2vD6QqBcArOmTX3t92exPzNNZQg=='}}], 'explanation': [{'type': 'text', 'text': 'This joke relies on a play on words (puns) involving pizza-related terms. Here is the breakdown:\n\n**1. "A pizza my heart"**\nThis is a pun on the common romantic phrase, **"a piece of my heart."** By using the word "pizza," the joke implies that the speaker is already romantically attached to the pizza, or that the pizza is "taken."\n\n**2. "Afraid of getting cheesy"**\nThis works on two levels:\n*   **The Literal Meaning:** Pizza is covered in cheese, and when it gets hot, it gets stringy or "cheesy."\n*   **The Figurative Meaning:** In English, if someo

In [10]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': [{'type': 'text',
   'text': 'Why did the pasta go to the doctor?\n\nBecause it was feeling **cannelloni**.',
   'extras': {'signature': 'EnEKbwFpFH0TBMsU3N+KqLC5uZsPHVZMbyW2glsBASJ59pWqt+sTpZgwcV/GWwrbPjSNVcM4xvYRR9PS/kqnU2q9uNo5uvkZVdRHBE2FKS+GnrACt9ZuWUjV50Q1gsMXj1wwoahq7GDWxhAjXN87kSXbqQ=='}}],
 'explanation': [{'type': 'text',
   'text': 'This joke relies on a **pun**, which is a play on words that sound similar but have different meanings.\n\nHere is the breakdown:\n\n1.  **The Phrase:** The pasta is feeling **"cannelloni."**\n2.  **The Sound-Alike:** "Cannelloni" is a type of Italian pasta shaped like large, hollow tubes. When spoken aloud, it sounds very similar to the phrase **"can only"** (as in, "it felt *can only*...").\n3.  **The Context:** The joke implies that the pasta was feeling "can only" (lonely). By replacing "can only" with "cannelloni," the joke creates a humorous connection between the pasta\'s emotional state and its name.\n\nIn shor

In [11]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'joke': [{'type': 'text', 'text': 'Why did the pasta go to the doctor?\n\nBecause it was feeling **cannelloni**.', 'extras': {'signature': 'EnEKbwFpFH0TBMsU3N+KqLC5uZsPHVZMbyW2glsBASJ59pWqt+sTpZgwcV/GWwrbPjSNVcM4xvYRR9PS/kqnU2q9uNo5uvkZVdRHBE2FKS+GnrACt9ZuWUjV50Q1gsMXj1wwoahq7GDWxhAjXN87kSXbqQ=='}}], 'explanation': [{'type': 'text', 'text': 'This joke relies on a **pun**, which is a play on words that sound similar but have different meanings.\n\nHere is the breakdown:\n\n1.  **The Phrase:** The pasta is feeling **"cannelloni."**\n2.  **The Sound-Alike:** "Cannelloni" is a type of Italian pasta shaped like large, hollow tubes. When spoken aloud, it sounds very similar to the phrase **"can only"** (as in, "it felt *can only*...").\n3.  **The Context:** The joke implies that the pasta was feeling "can only" (lonely). By replacing "can only" with "cannelloni," the joke creates a humorous connection between the pasta\'s emotional state and its name.\

In [12]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': [{'type': 'text', 'text': 'Why did the pasta go to the doctor?\n\nBecause it was feeling **cannelloni**.', 'extras': {'signature': 'EnEKbwFpFH0TBMsU3N+KqLC5uZsPHVZMbyW2glsBASJ59pWqt+sTpZgwcV/GWwrbPjSNVcM4xvYRR9PS/kqnU2q9uNo5uvkZVdRHBE2FKS+GnrACt9ZuWUjV50Q1gsMXj1wwoahq7GDWxhAjXN87kSXbqQ=='}}], 'explanation': [{'type': 'text', 'text': 'This joke relies on a **pun**, which is a play on words that sound similar but have different meanings.\n\nHere is the breakdown:\n\n1.  **The Phrase:** The pasta is feeling **"cannelloni."**\n2.  **The Sound-Alike:** "Cannelloni" is a type of Italian pasta shaped like large, hollow tubes. When spoken aloud, it sounds very similar to the phrase **"can only"** (as in, "it felt *can only*...").\n3.  **The Context:** The joke implies that the pasta was feeling "can only" (lonely). By replacing "can only" with "cannelloni," the joke creates a humorous connection between the pasta\'s emotional state and its name.

### Time Travel

In [15]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f1b3438-2f19-65e4-8000-e1fbfb97489f"}})

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1b3438-2f19-65e4-8000-e1fbfb97489f'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-09-18T09:29:56.624739+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b3438-2f13-61b4-bfff-d650738658bd'}}, tasks=(PregelTask(id='e6fa567d-9f94-a3fa-cb0c-896941e91c4d', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': [{'type': 'text', 'text': 'Why does a pizza never go out on a date?\n\nBecause it’s already got a **pizza** my heart, but it’s afraid of getting **cheesy**!', 'extras': {'signature': 'EnEKbwFpFH0TxkToFjXaeu9cbVNuCGFw6IsJ9MWsIcvLRWoJU+QzQW+1kpcSqw7CLIoCBdUscLV946JgSGPTXYaPvMkjERMLH68u4FVo3zTNK5hlOMzV8XBxl26IUfs2vD6QqBcArOmTX3t92exPzNNZQg=='}}]}),), interrupts=())

In [16]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f1b3438-2f19-65e4-8000-e1fbfb97489f"}})

{'topic': 'pizza',
 'joke': [{'type': 'text',
   'text': "Why did the pizza maker go into business?\n\nHe wanted to make some **dough**, but he couldn't handle the **pressure**, so he just kept **kneading** the money!",
   'extras': {'signature': 'EnEKbwFpFH0TiWQkiIsKp/wImTT5QKvr1E3tHwrx2TQPgDLBe/VbKfmXAqvTkzsPBA3YobTGOlzvgKFTheQ5CdkrR9YFc0gKow6HfuBaUOvR0Ow2vdTiMUsrUr06fqHa6SFhH0jg7fOxnMblMKNAs5IJrA=='}}],
 'explanation': [{'type': 'text',
   'text': 'This joke is a play on words (a pun) that uses baking terminology to describe the financial aspects of running a business. Here is the breakdown of the three puns used:\n\n1.  **"Make some dough":** In baking, "dough" is the mixture used to make pizza crust. In slang, "dough" is a common term for **money**. The pizza maker wanted to earn a profit.\n2.  **"Handle the pressure":** In the context of a business, "pressure" refers to the stress of managing operations and competition. In the context of pizza making, you have to apply physical *

In [17]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': "Why did the pizza maker go into business?\n\nHe wanted to make some **dough**, but he couldn't handle the **pressure**, so he just kept **kneading** the money!", 'extras': {'signature': 'EnEKbwFpFH0TiWQkiIsKp/wImTT5QKvr1E3tHwrx2TQPgDLBe/VbKfmXAqvTkzsPBA3YobTGOlzvgKFTheQ5CdkrR9YFc0gKow6HfuBaUOvR0Ow2vdTiMUsrUr06fqHa6SFhH0jg7fOxnMblMKNAs5IJrA=='}}], 'explanation': [{'type': 'text', 'text': 'This joke is a play on words (a pun) that uses baking terminology to describe the financial aspects of running a business. Here is the breakdown of the three puns used:\n\n1.  **"Make some dough":** In baking, "dough" is the mixture used to make pizza crust. In slang, "dough" is a common term for **money**. The pizza maker wanted to earn a profit.\n2.  **"Handle the pressure":** In the context of a business, "pressure" refers to the stress of managing operations and competition. In the context of pizza making, you have to apply

#### Updating State

In [18]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f1b3438-2f19-65e4-8000-e1fbfb97489f", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1b3443-cab6-6bf2-8001-51ddffc79e62'}}

In [19]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b3443-cab6-6bf2-8001-51ddffc79e62'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-09-18T09:35:08.220554+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b3438-2f19-65e4-8000-e1fbfb97489f'}}, tasks=(PregelTask(id='f778d893-f9a3-e79c-03da-00adf31af334', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': "Why did the pizza maker go into business?\n\nHe wanted to make some **dough**, but he couldn't handle the **pressure**, so he just kept **kneading** the money!", 'extras': {'signature': 'EnEKbwFpFH0TiWQkiIsKp/wImTT5QKvr1E3tHwrx2TQPgDLBe/VbKfmXAqvTkzsPBA3YobTGOlzvgKFTheQ5CdkrR9YFc0gKow6HfuBaUOvR0Ow2vdTiM

In [21]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc72-ca16-6359-8001-7eea05e07dd2"}})

EmptyInputError: Received no input for __start__

In [116]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'Why did the samosa bring a ladder to the party? \nBecause it wanted to be the best snack in the room and rise to the occasion!', 'explanation': 'This joke plays on the double meaning of the word "rise." In one sense, "rise" means to physically move upwards, which is why the samosa brought a ladder to the party. However, in another sense, "rise" can also mean to perform well or excel, as in rising to the occasion. So, the samosa brought a ladder to symbolize its desire to physically rise above the other snacks at the party and also to metaphorically rise to the occasion by being the best snack in the room.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc75-4407-6195-8003-b08dcfd27511'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}, 'thread_id': '1'}, created_at='2025-07-29T21:59:41.628661+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoin

### Fault Tolerance

In [22]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [23]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str
    step3: str

In [24]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [25]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)


In [1]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)


🔁 Re-running the graph to demonstrate fault tolerance...


NameError: name 'graph' is not defined

In [ ]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))